In [1]:
import pandas as pd
from scipy.stats import fisher_exact

In [2]:
# same rank as fig3_go_enrichment.ipynb
coemb = "../../result/HEK293T_HCT116/coemb"
df1 = pd.read_csv(
    f"{coemb}/promergeclub_HCT116anchor/HEK293T/"
    "HEK293T_specific_score.tsv",
    sep="\t",
    index_col=0,
)
df2 = pd.read_csv(
    f"{coemb}/promergeclub_HEK293Tanchor/HCT116/"
    "HCT116_specific_score.tsv",
    sep="\t",
    index_col=0,
)
df = (
    df1["specific_score"].rank(ascending=False)
    + df2["specific_score"].rank(ascending=False)
)
df = df.sort_values().dropna()
n_tail = int(len(df) * 0.1)

10635 1063


In [3]:
# map to the current HGNC symbol
hgnc = pd.read_csv(
    "../../data/HGNC/hgnc_complete_set.txt",
    sep="\t",
    low_memory=False,
)
ens2sym = (
    hgnc.dropna(subset=["ensembl_gene_id"])
    .drop_duplicates("ensembl_gene_id")
    .set_index("ensembl_gene_id")["symbol"]
)
tfdb = pd.read_csv(
    "../../data/HumanTFs/DatabaseExtract_v_1.01.csv",
    index_col=0,
)
tfdb = tfdb[tfdb["Is TF?"] == "Yes"]
tf = set(
    tfdb["Ensembl ID"].map(ens2sym)
    .fillna(tfdb["HGNC symbol"].str.strip())
)
coding = set(
    hgnc.loc[hgnc["locus_group"] == "protein-coding gene", "symbol"]
)
p_genome = len(tf & coding) / len(coding)

1639 1628 19297


In [4]:
# top 10% vs the rest of the rank, and the genome share
is_tf = pd.Series(df.index.isin(tf), index=df.index)
top = is_tf.iloc[:n_tail]
rest = is_tf.iloc[n_tail:]
odds, p = fisher_exact(
    [
        [top.sum(), (~top).sum()],
        [rest.sum(), (~rest).sum()],
    ]
)
print(f"top: {top.sum()} / {len(top)} TFs = {top.mean():.3f}")
print(f"rest: {rest.sum()} / {len(rest)} TFs = {rest.mean():.3f}")
print(f"odds ratio {odds:.2f}, Fisher's exact p = {p:.4f}")
print(f"genome: {p_genome:.3f}")

top: 88 / 1063 TFs = 0.083
rest: 577 / 9572 TFs = 0.060
odds ratio 1.41, Fisher's exact p = 0.0061
genome: 0.084
